# Lecture 02, Notebook 01: Lux-Native Foundations

This Julia/Lux translation keeps the classical supervised-learning idea from the
Python notebook, then crosses into Lux with feature-by-batch arrays and an
explicit `model(x, ps, st)` call.

## Lecture 02, Notebook 01: ML Foundations and Supervised Learning

**Course:** Deep Learning for Solving and Estimating Dynamic Models in Economics and Finance
**Script reference:** §1.1–1.3 (ML foundations: supervised learning, loss functions, clustering)
**Notebook role:** core
**Author:** Simon Scheidegger

*Julia/Lux preview of* `lectures/lecture_02_intro_deep_learning/code/lecture_02_01_BasicML_intro.ipynb`.

## Tutorial: Linear Regression, Classification, Unsupervised Learning, and Loss Functions

The Python ground-truth notebook illustrates four core machine-learning ideas:

1. **Linear Regression:** fit a linear model to synthetic data and measure prediction quality with MSE and MAE.
2. **Linear Classification:** train a logistic-regression classifier and visualise the decision boundary.
3. **Unsupervised Learning:** apply k-means clustering to discover natural groupings in unlabeled data.
4. **Loss Functions:** plot the standard losses for regression (squared error, absolute error) and classification (binary cross-entropy).

This compact Lux preview focuses on the **linear-regression** example, contrasting an ordinary-least-squares baseline with a small Lux MLP, and reproduces the regression/classification **loss gallery** as prose. Classification and clustering remain in the Python ground truth above.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", "..", "julia"))

using CairoMakie
using DLEFJulia
using LinearAlgebra
using Lux
using NNlib
using Optimisers
using Statistics

In [ ]:
RUN_MODE = "smoke"
SEED = 0
rng = rng_from_seed(SEED)
budget = run_mode_budget(RUN_MODE)

---
### 1. Linear Regression

**Goal:** predict a continuous target $y \in \mathbb{R}$ from a single feature $x$.

We generate synthetic data from the true model $y = a + b\,x + \varepsilon$, $\varepsilon \sim \mathcal{N}(0,\sigma^2)$, fit an ordinary-least-squares baseline, and compare it with a small Lux MLP trained by gradient descent. This preview uses rescaled toy constants ($n=80$, $a=1$, $b=2.5$, $\sigma=0.15$) rather than the Python notebook's $n=100$, $a=0$, $b=3$, $\sigma=1$ setup.

Rows are observations here; we convert to feature-by-batch arrays only at the Lux boundary, then call the model with the explicit `y, st = model(x, ps, st)` pattern.

In [ ]:
n = 80
x_scalar = collect(range(-2.0, 2.0; length = n))
noise = 0.15 .* randn(rng, n)
y_scalar = 1.0 .+ 2.5 .* x_scalar .+ noise

design = hcat(ones(n), x_scalar)
ols_coef = design \ y_scalar
y_ols = design * ols_coef
ols_metrics = (
    intercept = ols_coef[1],
    slope = ols_coef[2],
    mse = mse_loss(y_ols, y_scalar),
    mae = mae_loss(y_ols, y_scalar),
)

In [ ]:
features = to_feature_batch(reshape(x_scalar, :, 1))
targets = reshape(y_scalar, 1, :)
assert_matching_batch(features, targets)

model = make_mlp(1, (16, 16), 1; activation = NNlib.tanh)
ps, st = setup_model(rng_from_seed(SEED; offset = 1), model; parameter_type = Float64)
train_state = setup_training(model, ps, st, Optimisers.Adam(0.02))

supervised_loss(model, ps, st, batch) = begin
    prediction, st_new = model(batch.x, ps, st)
    return mse_loss(prediction, batch.y), st_new
end

batch = (x = features, y = targets)
initial_loss = loss_value(train_state, supervised_loss, batch)
history = NamedTuple[]
for _ in 1:budget.steps
    metrics = train_step!(train_state, supervised_loss, batch; max_grad_norm = 10.0)
    append_metric!(history; step = metrics.step, loss = metrics.loss)
end
final_loss = loss_value(train_state, supervised_loss, batch)

In [ ]:
y_lux, _ = train_state.model(features, train_state.ps, train_state.st)
fig = Figure(size = figure_size(RUN_MODE))
ax = Axis(fig[1, 1], xlabel = "x", ylabel = "y")
scatter!(ax, x_scalar, y_scalar; color = (:gray35, 0.55), label = "data")
lines!(ax, x_scalar, y_ols; color = :dodgerblue3, linewidth = 3, label = "OLS")
lines!(ax, x_scalar, vec(y_lux); color = :darkorange, linewidth = 3, label = "Lux MLP")
axislegend(ax; position = :lt)
fig

---
### Beyond regression: classification, clustering, and the loss gallery

The full Python notebook continues past regression:

- **Linear classification.** A logistic-regression model assigns each input $\mathbf{x}$ to one of $K$ classes, and its decision boundary is visualised.
- **Unsupervised learning.** K-means partitions $n$ observations into $k$ clusters by minimising the within-cluster sum of squared distances to the centroid — no labels required.

#### Loss functions

Every supervised-learning algorithm follows the same recipe:

1. Choose a model $\hat{y} = h(\mathbf{x};\boldsymbol{\theta})$
2. Define a loss $J(\boldsymbol{\theta})$
3. Optimise $\boldsymbol{\theta}^{*} = \arg\min J(\boldsymbol{\theta})$

Regression losses (used above via `mse_loss` and `mae_loss`):

| Loss | Per-sample formula |
|---|---|
| Squared error (MSE) | $(y - \hat{y})^{2}$ |
| Absolute error (MAE) | $\lvert y - \hat{y}\rvert$ |

Classification loss — binary cross-entropy:

$$
J = -\Bigl[y\,\log\hat{y} \;+\; (1-y)\,\log(1-\hat{y})\Bigr],
\qquad y\in\{0,1\},\;\hat{y}\in(0,1)
$$

This is the standard loss used in logistic regression and neural-network classifiers.

---
### Conclusion

In this Lux preview we:

- **Linear Regression:** fitted an OLS baseline and a small Lux MLP to synthetic data, measuring quality with MSE and MAE.
- **Classification & Clustering:** reviewed the logistic-regression and k-means examples carried in full by the Python ground truth.
- **Loss Functions:** summarised the squared-error, absolute-error, and binary cross-entropy losses.

These building blocks — models, losses, and optimisation — reappear throughout the course as we move to deep neural networks. The cell below returns a machine-checkable summary of this notebook's run.

In [ ]:
(
    ols = ols_metrics,
    lux_initial_loss = initial_loss,
    lux_final_loss = final_loss,
    steps = length(history),
)